# 框架运行时、数据与性能补充线 · 第 7/8 课：Profiler、Timeline 与 CUDA Allocator

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现区间并集计算暴露时间，正确区分 allocated/reserved、碎片和真实泄漏。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/lesson12` 给出 MFU；本课训练读取 trace 的因果关系，并用 allocator snapshot 解释 OOM。

前置：Python、PyTorch、train 第 1～5 课、CUDA 基础。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Profiler 记录 CPU op、runtime launch、memcpy 和 GPU kernel 区间。重叠区间不能简单相加；allocator 的 allocated 是活 tensor，reserved 是缓存池持有，inactive split 可提示碎片。

### 数据与控制如何流动

先在短且代表性的 warmup/active 窗口采 trace，再沿 CPU op→CUDA runtime→stream kernel/memcpy 关联空洞；内存问题则同时比较 allocated/reserved 曲线和 snapshot 中 segment/block 生命周期。

### 正确性条件与常见误区

异步 GPU 必须用 event/同步的正确计时；`empty_cache()` 只释放未使用缓存给系统，不会释放活 tensor，也不应作为每 step 修复。

### 性能、成本与工程取舍

record_shapes/stack/memory 有观测开销；短窗口和 warmup 可减少扰动。大 reserved-allocated 可能是缓存或碎片，要结合 snapshot 生命周期判断。

## 具体演示

GPU kernels 区间 [0,4]、[3,7] 总忙时是 7 而非 8；若 step=10，暴露空闲 3。错误相加会得到不可能的 80%+指标。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐半开区间并集时长；相交或相邻区间应合并。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def union_duration(intervals):
    if any(end < start for start, end in intervals):
        raise ValueError("negative interval")
    if not intervals:
        return 0
    ordered = sorted(intervals)
    total = 0
    start, end = ordered[0]
    for next_start, next_end in ordered[1:]:
        if next_start <= end:
            end = max(end, next_end)
        else:
            total += end - start
            start, end = next_start, next_end
    # TODO：循环结束后最后一段尚未计入。
    return ______

assert union_duration([(0, 4), (3, 7), (9, 10)]) == 8
assert union_duration([]) == 0


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

`memory_reserved` 高于 `memory_allocated` 是否等于内存泄漏？

**你的答案：**


### Q2

为什么 CPU wall-clock 包围 CUDA op 会低估或高估单 kernel 时间？

**你的答案：**


### Q3

trace 中 GPU 空闲但 CPU 也不忙，下一步查什么？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [torch.profiler](https://docs.pytorch.org/docs/stable/profiler.html)
- [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [Understanding CUDA memory usage](https://docs.pytorch.org/docs/stable/torch_cuda_memory.html)

API 与平台能力会演进；部署前应按目标版本重新核对。